# Poisson-HMM state analysis: auditory cortex vs. CP, with null-data and cross-stimulus controls

This notebook copies the session loading / unit and trial setup from `decoder_v1.ipynb`, then:

1. Fits one Poisson HMM per stimulus (pooled across `rewarded_modality` task contexts) for the **auditory cortex** units (`area1_units`, `location` contains `'AUD'`), each compared against a matched **null-data control** (surrogate spike trains from a known continuous, single-state process with no true jump anywhere -- see `hmm_null_control.py`).
2. Repeats the same per-stimulus fit + null control for the **CP (striatum)** units (`area2_units`, `location == 'CP'`).
3. Plots each trial's dominant HMM state over time, with trials sorted and color-coded by behavioral outcome (**hit** / **correct reject** / **incorrect**).
4. Runs a **cross-stimulus generalization test**: fits an HMM on one stimulus's trials only, then applies those *fixed* parameters (forward-backward, no re-fitting) to the complementary stimulus's held-out trials, to check whether the recovered state transition is a genuine, stimulus-general population dynamic or an artifact of fitting each stimulus separately.

All fitting/decoding goes through `hmm_helpers.py`; null-data generation goes through `hmm_null_control.py`; the per-stimulus/cross-generalization orchestration and outcome-aware plotting live in `hmm_stim_null_generalization.py`.


## 1. Initialization and data loading (copied from `decoder_v1.ipynb`)


In [1]:
# Standard library
import os
from pathlib import Path  # Object-oriented filesystem paths

# Data handling packages
import numpy as np
import numpy.random as npr
import pandas as pd
import pynwb
import ipywidgets as widgets
from IPython.display import display
import gc
import time
import sklearn

# Progress bar utility
from tqdm import tqdm  # Displays a smart progress bar during loops

from IPython.display import Image, display
import glob

# Preprocessing
from sklearn.preprocessing import StandardScaler  # Standardizes features (zero mean, unit variance)
from sklearn.svm import LinearSVC
from sklearn import svm
from sklearn.model_selection import train_test_split, cross_validate
from sklearn.metrics import confusion_matrix

# Plotting libraries
import matplotlib.pyplot as plt
from matplotlib import colors
import seaborn as sns

# HMM-related imports from JAX, Dynamax, and TensorFlow Probability
from functools import partial
import jax.numpy as jnp
import jax.random as jr
from dynamax.hidden_markov_model import GaussianHMM
from dynamax.utils.utils import find_permutation

# Additional HMM variants and plotting utilities from Dynamax
from dynamax.hidden_markov_model import (
    DiagonalGaussianHMM,
    SphericalGaussianHMM,
    SharedCovarianceGaussianHMM
)

from dynamax.utils.plotting import CMAP, COLORS, white_to_color_cmap

# Pandas display settings
pd.set_option('display.max_columns', None)  # Ensures all columns are shown when printing DataFrames

# Inline plotting for Jupyter Notebooks
%matplotlib inline


In [2]:
root = Path("/root/capsule/data/dynamicrouting_datacube")

example_session_ids = [
                        "759434_2025-02-04", "713655_2024-08-09", "743199_2024-12-05",
                        "712815_2024-05-22", "742903_2024-10-23", "664851_2023-11-16",
                        "741137_2024-10-10", "662892_2023-08-24", "714748_2024-06-24",
                        "667252_2023-09-28", "715710_2024-07-16", "708016_2024-04-29",
                    ]


session_id = example_session_ids[1] #session 4? mouse 4?

# loop through the directories to find the NWB file for the specific session
for d in root.iterdir():
    nwb_path = d / f"{session_id}.nwb.zarr"
    if nwb_path.exists():
        print(nwb_path)
        break


/root/capsule/data/dynamicrouting_datacube/ecephys_713655_2024-08-09_10-41-47_nwb_2026-08-04_15-02-39/713655_2024-08-09.nwb.zarr


In [3]:
session = pynwb.read_nwb(nwb_path)


/opt/conda/lib/python3.12/site-packages/hdmf_zarr/backend.py:1699: UserWarning: Inferred dtype from zarr type. Dataset missing zarr_dtype: data   <zarr.core.Array '/processing/behavior/facemap_front_camera/data' (460263, 500) float32 read-only>
  warnings.warn(
/opt/conda/lib/python3.12/site-packages/hdmf_zarr/backend.py:1699: UserWarning: Inferred dtype from zarr type. Dataset missing zarr_dtype: data   <zarr.core.Array '/processing/behavior/facemap_side_camera/data' (460280, 500) float32 read-only>
  warnings.warn(


In [4]:
# QC thresholds from Shawn

units_table = session.units.to_dataframe()


MAX_ISI_VIOLATIONS_RATIO = 0.5    # lower is better
MAX_AMPLITUDE_CUTOFF = 0.1        # lower is better
MIN_PRESENCE_RATIO = 0.95         # higher is better
MAX_ACTIVITY_DRIFT = 0.2 # lower is better, moderate threshold

good_units = units_table[
    (units_table.isi_violations_ratio < MAX_ISI_VIOLATIONS_RATIO) &
    (units_table.amplitude_cutoff < MAX_AMPLITUDE_CUTOFF) &
    (units_table.presence_ratio > MIN_PRESENCE_RATIO) &
    (units_table.activity_drift < MAX_ACTIVITY_DRIFT)
]
print('%d of %d units passed quality control' % (len(good_units), len(units_table)))


1365 of 3577 units passed quality control


In [5]:
trials = session.trials[:]
trials.head()


,start_time,stop_time,quiescent_start_time,quiescent_stop_time,stim_start_time,stim_stop_time,response_window_start_time,response_window_stop_time,task_control_response_time,response_time,reward_time,post_response_window_start_time,post_response_window_stop_time,stim_name,grating_phase,block_index,rewarded_modality,trial_index,trial_index_in_block,repeat_index,is_response,is_correct,is_incorrect,is_hit,is_false_alarm,is_correct_reject,is_miss,is_go,is_nogo,is_rewarded,is_noncontingent_reward,is_contingent_reward,is_reward_scheduled,is_instruction,is_aud_stim,is_vis_stim,is_catch,is_target,is_aud_target,is_vis_target,is_nontarget,is_aud_nontarget,is_vis_nontarget,is_vis_rewarded,is_aud_rewarded,is_block_switch,is_repeat,is_opto,is_task_control_correct
id,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
0,2403.42415,2408.92875,2403.42415,2404.89204,2404.93261,2405.43261,2405.00879,2405.92623,2405.47583,2405.46553,2405.49254,2405.953890,2408.956420,sound1,NaN,0,aud,0,0.0,0.0,True,True,False,True,False,False,False,True,False,True,False,True,True,True,True,False,False,True,True,False,False,False,False,False,True,False,False,False,False
1,2409.89623,2415.40125,2409.89623,2411.36411,2411.40501,2411.90501,2411.48088,2412.39832,2411.74797,2411.73257,2411.76447,2412.426005,2415.428515,sound1,NaN,0,aud,1,1.0,0.0,True,True,False,True,False,False,False,True,False,True,False,True,True,True,True,False,False,True,True,False,False,False,False,False,True,False,False,False,False
2,2417.98632,2423.49098,2417.98632,2419.45423,2419.49515,2419.99515,2419.57102,2420.48843,2420.03805,2420.03365,2420.05473,2420.516120,2423.518630,sound1,NaN,0,aud,2,2.0,0.0,True,True,False,True,False,False,False,True,False,True,False,True,True,True,True,False,False,True,True,False,False,False,False,False,True,False,False,False,False
3,2429.22911,2434.73372,2429.22911,2430.69701,2430.73779,2431.23779,2430.81378,2431.73118,2431.43094,2431.43076,2431.44765,2431.758880,2434.761390,sound1,NaN,0,aud,3,3.0,0.0,True,True,False,True,False,False,False,True,False,True,False,True,True,True,True,False,False,True,True,False,False,False,False,False,True,False,False,False,False
4,2435.43427,2440.93889,2435.43427,2436.90215,2436.94313,2437.44313,2437.01895,2437.93632,2437.60277,2437.60187,2437.61948,2437.964090,2440.966610,sound1,NaN,0,aud,4,4.0,0.0,True,True,False,True,False,False,False,True,False,True,False,True,True,True,True,False,False,True,True,False,False,False,False,False,True,False,False,False,False


In [6]:
key1 = 'AUD'
key2 = 'CP'


area1_units = good_units[good_units['location'].str.contains(key1)]
area2_units = good_units[good_units['location'].str.contains(key2)]
print(f'{len(area1_units)} good {key1} units, {len(area2_units)} good {key2} units')


173 good AUD units, 98 good CP units


## 2. HMM / null-control / cross-generalization setup

`hmm_stim_null_generalization.py` wraps `hmm_helpers.py` (fitting/decoding) and
`hmm_null_control.py` (surrogate null spike-train generation) into three pieces
used throughout this notebook:

- `run_area_stimulus_analysis` -- one HMM per stimulus, pooled across contexts, plus a matched null-control fit.
- `cross_generalize_stimulus_pair` -- fit on one stimulus, decode a held-out complementary stimulus with fixed parameters.
- `plot_trial_state_raster_with_outcomes` / `plot_cross_generalization_comparison` -- outcome-aware plotting.


In [7]:
import sys
if "/root/capsule/code" not in sys.path:
    sys.path.insert(0, "/root/capsule/code")

from hmm_stim_null_generalization import (
    run_area_stimulus_analysis,
    cross_generalize_stimulus_pair,
    plot_trial_state_raster_with_outcomes,
    plot_cross_generalization_comparison,
    trial_outcome_labels,
    OUTCOME_COLORS,
)

# --- binning window (matches the ACx PoissonHMM analysis in get_hmmstates.ipynb) ---
BIN_WIDTH = 0.010            # 10 ms bins
T_PRE_S, T_POST_S = -0.5, 1.5  # seconds relative to stim_start_time
bin_edges = np.arange(T_PRE_S, T_POST_S + BIN_WIDTH, BIN_WIDTH)
bin_centers = bin_edges[:-1] + BIN_WIDTH / 2

# --- HMM fitting config ---
NUM_STATES = 4
N_RESTARTS = 5
NUM_ITERS = 60
SEED = 0


## 3. Auditory cortex (`area1_units`): per-stimulus HMM fits vs. null control

One HMM is fit per stimulus (`sound1`, `sound2`, `vis1`, `vis2`), pooling trials across
both `rewarded_modality` context blocks -- i.e. all trials of a given physical stimulus,
regardless of which block they occurred in. Catch trials are excluded. Each fit is
compared against a matched null-control fit on surrogate data with no true state change
(see `hmm_null_control.simulate_null_spike_trains`).


In [ ]:
auditory_results = run_area_stimulus_analysis(
    area1_units, trials, 'Auditory cortex (AUD)', bin_edges,
    num_states=NUM_STATES, n_restarts=N_RESTARTS, num_iters=NUM_ITERS, seed=SEED,
)



=== Auditory cortex (AUD): stimulus "sound1" (pooled across contexts) ===
  118 trials, 173 units
  real fit LL = -557601.6
  null-control fit LL = -572496.2

=== Auditory cortex (AUD): stimulus "sound2" (pooled across contexts) ===
  109 trials, 173 units
  real fit LL = -493676.5


In [ ]:
def summarize_and_plot_area_results(results, area_label):
    """For each stimulus's fit, prints a real-vs-null onset/confidence summary
    and plots the real and null dominant-state trial rasters side by side,
    colored by trial outcome."""
    for stim, r in results.items():
        real_found = r['real_onsets'][~np.isnan(r['real_onsets'])]
        null_found = r['null_onsets'][~np.isnan(r['null_onsets'])]
        real_conf = r['real']['probs'][:, :, -1].max(axis=1)
        null_conf = r['null']['probs'][:, :, -1].max(axis=1)

        print(f"\n--- {area_label} | {stim} ---")
        print(f"  real: LL={r['real']['ll']:.1f}, "
              f"frac trials with late-state onset={len(real_found)/len(r['real_onsets']):.2f}, "
              f"median confidence={np.median(real_conf):.2f}")
        print(f"  null: LL={r['null']['ll']:.1f}, "
              f"frac trials with late-state onset={len(null_found)/len(r['null_onsets']):.2f}, "
              f"median confidence={np.median(null_conf):.2f}")

        fig, axes = plt.subplots(1, 2, figsize=(16, max(4, 0.05 * r['counts'].shape[0])), dpi=130)
        plot_trial_state_raster_with_outcomes(
            r['real']['probs'], r['outcome_labels'], r['bin_centers'], NUM_STATES,
            title=f'{area_label} | {stim} -- REAL', ax=axes[0])
        plot_trial_state_raster_with_outcomes(
            r['null']['probs'], r['outcome_labels'], r['bin_centers'], NUM_STATES,
            title=f'{area_label} | {stim} -- NULL CONTROL', ax=axes[1])
        plt.tight_layout()
        plt.show()


summarize_and_plot_area_results(auditory_results, 'AUD')


## 4. CP / striatum (`area2_units`): per-stimulus HMM fits vs. null control

Same per-stimulus, pooled-across-context fitting procedure, applied to the CP units.


In [ ]:
cp_results = run_area_stimulus_analysis(
    area2_units, trials, 'CP (striatum)', bin_edges,
    num_states=NUM_STATES, n_restarts=N_RESTARTS, num_iters=NUM_ITERS, seed=SEED,
)


In [ ]:
summarize_and_plot_area_results(cp_results, 'CP')


## 5. Cross-stimulus generalization test

Each per-stimulus HMM above was fit and decoded on the SAME trials, which cannot rule out
overfitting: an HMM with enough states can lock onto stimulus-specific structure that has
nothing to do with a shared, meaningful population dynamic. To test this, an HMM is fit on
one stimulus's trials only (e.g. `sound1`), and those FIXED parameters -- transition matrix
and emission rates, no re-fitting -- are used to decode the COMPLEMENTARY stimulus's
held-out trials (e.g. `sound2`) via the forward-backward algorithm
(`hmm_helpers.compute_posterior_state_probabilities`).

If the state transition is a genuine, stimulus-general population dynamic, it should still
be recovered on the held-out stimulus with a similarly-timed onset latency and a similar
fraction of trials crossing threshold. A large latency shift, or a collapse in how many
trials ever reach the threshold, means the original fit was stimulus-specific rather than
finding a shared state -- another way the pipeline can fool itself, alongside the null
control above.


In [ ]:
def run_and_plot_cross_generalization(units_df, area_label, stim_a, stim_b):
    """Fits on stim_a only, decodes held-out stim_b with those fixed
    parameters, and plots the onset-latency comparison in both directions
    (stim_a -> stim_b and stim_b -> stim_a)."""
    cg = cross_generalize_stimulus_pair(
        units_df, trials, stim_a, stim_b, bin_edges,
        num_states=NUM_STATES, n_restarts=N_RESTARTS, num_iters=NUM_ITERS, seed=SEED,
    )
    print(f"\n--- {area_label}: fit on {stim_a}, cross-decode held-out {stim_b} ---")
    print(f"  {stim_a} native fit LL={cg['fit_a']['ll']:.1f}; "
          f"{stim_b} native fit LL={cg['fit_b_native']['ll']:.1f}")

    fig, axes, stats = plot_cross_generalization_comparison(
        cg['native_a_onsets'], cg['cross_b_onsets'],
        native_name=f'{stim_a} (native fit)',
        cross_name=f'{stim_b} (decoded w/ {stim_a} params)',
        title=f'{area_label}: {stim_a} -> {stim_b} cross-generalization',
    )
    plt.show()
    print(f"  stats: {stats}")

    # for reference: how does cross-decoding compare to just fitting stim_b natively?
    fig2, axes2, stats2 = plot_cross_generalization_comparison(
        cg['native_b_onsets'], cg['cross_b_onsets'],
        native_name=f'{stim_b} (native fit)',
        cross_name=f'{stim_b} (decoded w/ {stim_a} params)',
        title=f'{area_label}: {stim_b} native vs. cross-decoded from {stim_a}',
    )
    plt.show()
    print(f"  stats (native {stim_b} vs. cross-decoded): {stats2}")
    return cg


### Auditory cortex: `sound1` &harr; `sound2`


In [ ]:
cg_aud_sound = run_and_plot_cross_generalization(area1_units, 'AUD', 'sound1', 'sound2')
cg_aud_vis = run_and_plot_cross_generalization(area1_units, 'AUD', 'vis1', 'vis2')


### CP: `sound1` &harr; `sound2` and `vis1` &harr; `vis2`


In [ ]:
cg_cp_sound = run_and_plot_cross_generalization(area2_units, 'CP', 'sound1', 'sound2')
cg_cp_vis = run_and_plot_cross_generalization(area2_units, 'CP', 'vis1', 'vis2')
